# 06 - Add the original dataset

The competition data was made by a computer program, based on a real 10,000-row dataset:
**[EV Adoption Behavior & Range Anxiety](https://www.kaggle.com/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety)**
(file `original_data/EV_Adoption_and_Range_Anxiety_Dataset.csv`).

It has the same 13 columns and the same target. Differences: about 2% of its number values are missing, and it has wider value ranges.

We test 3 ways to use it. Each one uses the same tuned LightGBM, the same folds and learning rate 0.05:

| Way | What we do |
|---|---|
| **A. Add rows** | Add the 10k original rows to the **training part of each fold only**. They never go in the validation part, so the CV score stays honest. |
| **B. Hint columns** | For each column, add a new column: the "Yes" rate for that value **in the original data**. Only the original data's answers are used, so no train answers leak in. |
| **C. Both** | A + B together. |

The best way is saved as `lgbm_orig`. Then notebook 05 is re-run so the blend can include it.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from config import *
from utils import *

In [2]:
train, test, sample = load_data()
folds = get_folds(train)
y = train[TARGET].values

orig = pd.read_csv(ORIG_FILE).drop(columns=["Buyer_ID"])
orig[TARGET] = (orig[TARGET] == "Yes").astype(int)
print("original:", orig.shape, f" Yes rate: {orig[TARGET].mean():.4f}")
print("Missing values:", orig.isna().sum()[orig.isna().sum() > 0].to_dict())

p = ROOT / "best_lgbm_params.json"
PARAMS = {
    "objective": "binary", "learning_rate": 0.05, "n_estimators": 5000, "bagging_freq": 1,
    "random_state": SEED, "verbose": -1, "n_jobs": -1,
    **json.loads(p.read_text()),
}

original: (10000, 14)  Yes rate: 0.1750
Missing values: {'Annual_Income_USD': 178, 'Daily_Commute_km': 181, 'Environmental_Concern_Level': 184}


## How good is the original data on its own?
A quick 5-fold CV on the 10k original rows only. A very high score means the original target follows a simple rule of the columns.

In [3]:
from sklearn.model_selection import cross_val_score
o = to_category(orig[FEATURES])
a = cross_val_score(lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1),
                    o, orig[TARGET], cv=5, scoring="roc_auc").mean()
print(f"CV AUC on original data alone: {a:.4f}")

CV AUC on original data alone: 0.8938


## Build the "hint" columns (way B)
For each column: the "Yes" rate for each value, measured on the original data.
Values that never appear in the original data (common for income and commute) are left empty. LightGBM handles empty values.

In [4]:
def add_orig_hints(df):
    df = df.copy()
    cols = []
    for c in FEATURES:
        rate = orig.groupby(c)[TARGET].mean()
        df[f"orig_rate_{c}"] = df[c].map(rate).astype(float)
        cols.append(f"orig_rate_{c}")
    return df, cols

tr_h, HINT_COLS = add_orig_hints(train)
te_h, _ = add_orig_hints(test)
or_h, _ = add_orig_hints(orig)
print("Share of rows with a hint value (train):")
print(tr_h[HINT_COLS].notna().mean().round(3).to_string())

Share of rows with a hint value (train):
orig_rate_Age                            1.000
orig_rate_Annual_Income_USD              0.979
orig_rate_Daily_Commute_km               1.000
orig_rate_Number_of_Cars_Owned           1.000
orig_rate_Charging_Stations_Near_Home    1.000
orig_rate_Charging_Stations_Near_Work    1.000
orig_rate_Environmental_Concern_Level    1.000
orig_rate_Gender                         1.000
orig_rate_City_Type                      1.000
orig_rate_Current_Car_Type               1.000
orig_rate_Home_Charging_Possible         1.000
orig_rate_Subsidy_Available              1.000
orig_rate_Range_Anxiety_Level            1.000


## Training loop that can add the original rows to each fold's training part

In [5]:
def run_cv(cols, add_rows, label):
    X = to_category(tr_h[cols]); X_test = to_category(te_h[cols]); X_orig = to_category(or_h[cols])
    # keep the same category list everywhere, so codes line up
    for c in CAT_COLS:
        if c in cols:
            cats = X[c].cat.categories
            X_test[c] = X_test[c].cat.set_categories(cats)
            X_orig[c] = X_orig[c].cat.set_categories(cats)

    oof, test_pred = np.zeros(len(X)), np.zeros(len(X_test))
    for k in range(N_FOLDS):
        tr, va = folds != k, folds == k
        X_tr, y_tr = X[tr], y[tr]
        if add_rows:
            X_tr = pd.concat([X_tr, X_orig], ignore_index=True)
            y_tr = np.concatenate([y_tr, orig[TARGET].values])
        m = lgb.LGBMClassifier(**PARAMS).fit(
            X_tr, y_tr, eval_set=[(X[va], y[va])], eval_metric="auc",
            callbacks=[lgb.early_stopping(200, verbose=False)])
        oof[va] = m.predict_proba(X[va])[:, 1]
        test_pred += m.predict_proba(X_test)[:, 1] / N_FOLDS
    auc, ll = score(y, oof)
    print(f"{label:<18} AUC = {auc:.5f}   logloss = {ll:.5f}")
    return {"auc": auc, "ll": ll, "oof": oof, "test": test_pred}

res = {}
res["raw"] = run_cv(FEATURES, False, "raw (no original)")
res["A"] = run_cv(FEATURES, True, "A. add rows")
res["B"] = run_cv(FEATURES + HINT_COLS, False, "B. hint columns")
res["C"] = run_cv(FEATURES + HINT_COLS, True, "C. both")

raw (no original)  AUC = 0.94280   logloss = 0.22473


A. add rows        AUC = 0.94279   logloss = 0.22474


B. hint columns    AUC = 0.94288   logloss = 0.22459


C. both            AUC = 0.94273   logloss = 0.22483


## Keep the best way
It must beat "raw" to count as a gain.

In [6]:
best = max(["A", "B", "C"], key=lambda k: res[k]["auc"])
gain = res[best]["auc"] - res["raw"]["auc"]
print(f"Best way: {best}   gain over raw: {gain:+.5f}")

r = res[best]
save_preds("lgbm_orig", r["oof"], r["test"])
log_experiment("lgbm_orig", f"raw+orig way {best}", r["auc"], r["ll"], f"original dataset, way {best}, lr 0.05")
make_submission(r["test"], "lgbm_orig", sample)
pd.read_csv(EXPERIMENTS_FILE).sort_values("cv_auc", ascending=False)

Best way: B   gain over raw: +0.00007


Saved lgbm_orig.csv  rows=286571


,model,features,cv_auc,cv_logloss,notes,date
10,lgbm_orig,raw+orig way B,0.942877,0.224594,"original dataset, way B, lr 0.05",2026-10-17 18:07
8,blend_rank,lgbm_optuna+lgbm_fe,0.942846,NaN,ensemble,2026-10-17 16:42
7,blend_mean,lgbm_optuna+lgbm_fe,0.942844,NaN,ensemble,2026-10-17 16:42
9,blend_stack,lgbm_optuna+lgbm_fe,0.942842,0.224661,ensemble,2026-10-17 16:42
4,lgbm_optuna,raw,0.942828,0.224683,"optuna 46 trials, lr 0.02",2026-10-17 15:59
5,lgbm_fe,raw+A,0.942806,0.224730,lgbm with kept feature groups,2026-10-17 16:12
2,xgb_v1,raw,0.941857,0.226431,default params,2026-10-17 15:11
6,cat_fe,raw+A,0.941778,0.226582,catboost with kept feature groups,2026-10-17 16:39
3,cat_v1,raw,0.941772,0.226594,default params,2026-10-17 15:39
1,lgbm_v1,raw,0.941771,0.226571,default params,2026-10-17 15:09
